# try/except при загрузке CSV

В отчёте пайплайн ломается не на `fit`, а на «файл не там» / пустая таблица / нет столбца. Сегодня — устойчивая загрузка.

In [ ]:
from pathlib import Path
import pandas as pd


def load_trips(path: Path) -> pd.DataFrame:
    """Загрузить CSV; при отсутствии файла — FileNotFoundError с понятным текстом."""
    pass


def clean_trips(raw: pd.DataFrame) -> pd.DataFrame:
    """Удалить строки без distance_km или duration_min."""
    pass


def assert_usable(frame: pd.DataFrame) -> None:
    """Проверить: непустой df и есть столбцы distance_km, duration_min."""
    pass


## 1. Успешная загрузка

Реализуйте `load_trips` и загрузите существующий `trips.csv`.

**How:** `try` вокруг `pd.read_csv`; при успехе вернуть DataFrame.

In [ ]:
TRIPS_PATH = None
for p in (Path('trips.csv'), Path('../../data/trips.csv'), Path('../data/trips.csv')):
    if p.exists():
        TRIPS_PATH = p.resolve()
        break
assert TRIPS_PATH is not None
df = load_trips(TRIPS_PATH)
assert isinstance(df, pd.DataFrame)
assert len(df) > 0
print(df.shape)

## 2. Ошибка пути

Вызовите `load_trips` на несуществующем пути и поймайте `FileNotFoundError`.

**Pitfall:** голое `except:` глотает SyntaxError — ловите конкретный тип. В сообщении должны быть путь и подсказка.

In [ ]:
caught = False
try:
    load_trips(Path('no_such_file.csv'))
except FileNotFoundError as e:
    caught = True
    print('Поймали:', e)
assert caught
ERROR_MSG_NOTE = ''
assert len(ERROR_MSG_NOTE) > 20
print(ERROR_MSG_NOTE)

## 3. Очистка

`clean_trips`: удалить строки без `distance_km` или `duration_min` (`dropna(subset=...)`). Проверка: одна NaN в duration → на одну строку меньше.

In [ ]:
raw = df.copy()
raw.loc[raw.index[0], 'duration_min'] = None
cleaned = clean_trips(raw)
assert len(cleaned) == len(raw) - 1
print(len(raw), len(cleaned))

## 4. assert_usable

Пустой df и df без `duration_min` должны давать `ValueError`. Поймайте оба случая и напечатайте сообщение.

In [ ]:
empty = pd.DataFrame(columns=['distance_km', 'duration_min'])
try:
    assert_usable(empty)
    empty_ok = False
except ValueError as e:
    empty_ok = True
    print('empty:', e)
bad = pd.DataFrame({'distance_km': [1.0, 2.0], 'hour': [8, 9]})
try:
    assert_usable(bad)
    bad_ok = False
except ValueError as e:
    bad_ok = True
    print('bad cols:', e)
assert empty_ok and bad_ok

## 5. Pipeline до модели

`load_trips` → `clean_trips` → `assert_usable`. Выведите shape итога.

In [ ]:
pipeline_df = clean_trips(load_trips(TRIPS_PATH))
assert_usable(pipeline_df)
assert len(pipeline_df) > 0
print('pipeline', pipeline_df.shape)

## 6. Checkpoint: зачем try до fit

Одна фраза: почему ловить отсутствие файла **до** обучения модели, а не надеяться на ошибку внутри sklearn?

In [ ]:
WHY_TRY_FIRST = ''
assert len(WHY_TRY_FIRST) > 25
print(WHY_TRY_FIRST)